# Sports Image Classification
**CS 372 — Introduction to Applied Machine Learning, Spring 2026**

This notebook trains and evaluates deep learning models to classify images across 100 sports categories. We fine-tune pretrained EfficientNet-B0 and ResNet50 models using transfer learning, apply data augmentation and regularization, perform error analysis, and deploy the best model as a Gradio web application.

**Dataset:** [Sports Classification (gpiosenka)](https://www.kaggle.com/datasets/gpiosenka/sports-classification) — 13,492 training images across 100 sport classes, with 500 validation and 500 test images.

---

## 1. Setup & Dependencies

In [ ]:
!pip install torch torchvision gradio timm scikit-learn -q
print("Dependencies installed.")

Dependencies installed.


In [ ]:
import os
import torch
import torchvision
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from collections import Counter
from sklearn.metrics import f1_score, precision_score, recall_score
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## 2. Data Loading

The dataset is stored in Google Drive and unzipped to `/content/sports`. It comes pre-split into train/validation/test sets (13,492 / 500 / 500 images).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!unzip -q "/content/drive/MyDrive/sports-classification.zip" -d /content/sports
print("Dataset ready.")

ValueError: mount failed

In [ ]:
# Locate train/val/test directories
data_dir = '/content/sports'
for root, dirs, files_list in os.walk(data_dir):
    if 'train' in dirs:
        data_dir = root
        break

train_dir = os.path.join(data_dir, 'train')
val_dir   = os.path.join(data_dir, 'valid')
test_dir  = os.path.join(data_dir, 'test')

sports = os.listdir(train_dir)
print(f"Number of sports classes: {len(sports)}")
print(f"Sample classes: {sports[:5]}")

## 3. Data Preprocessing & Augmentation

Training images are augmented with random horizontal flips, rotation, and color jitter to improve generalization and reduce overfitting. All images are resized to 224×224 and normalized using ImageNet mean and standard deviation, which is standard practice for pretrained models.

In [ ]:
# Training transforms with augmentation
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Validation/test transforms (no augmentation)
val_test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load datasets
train_dataset = datasets.ImageFolder(train_dir, transform=train_transforms)
val_dataset   = datasets.ImageFolder(val_dir,   transform=val_test_transforms)
test_dataset  = datasets.ImageFolder(test_dir,  transform=val_test_transforms)

# DataLoaders with batching and shuffling
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=2)

class_names = train_dataset.classes

print(f"Train:   {len(train_dataset)} images")
print(f"Val:     {len(val_dataset)} images")
print(f"Test:    {len(test_dataset)} images")
print(f"Classes: {len(class_names)}")

## 4. Baseline Model

Before training any neural network, we establish a random baseline. With 100 classes, random guessing should achieve approximately 1% accuracy.

In [ ]:
def random_baseline(loader, num_classes=100):
    correct, total = 0, 0
    for _, labels in loader:
        preds = torch.randint(0, num_classes, (labels.size(0),))
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return correct / total

baseline_acc = random_baseline(val_loader)
print(f"Random baseline accuracy: {baseline_acc:.2%} (expected ~1% for 100 classes)")

## 5. Model Training

We define a reusable `train_and_save` function that trains any model and saves the best checkpoint (by validation accuracy) to Google Drive. This prevents data loss on Colab session resets.

**Regularization techniques applied:**
- Dropout (built into EfficientNet-B0 architecture)
- L2 weight decay via AdamW optimizer (`weight_decay=1e-4`)
- Early stopping via best-checkpoint saving
- Gradient clipping (`max_norm=1.0`) for training stability

**Learning rate scheduling:** CosineAnnealingLR gradually reduces the learning rate over training epochs, allowing the optimizer to settle into a better minimum.

In [ ]:
EFFICIENTNET_PATH = '/content/drive/MyDrive/best_model.pth'
RESNET_PATH       = '/content/drive/MyDrive/best_resnet.pth'

criterion = nn.CrossEntropyLoss()

def train_and_save(model, optimizer, scheduler, save_path, epochs, name):
    """Train a model, saving the best checkpoint by validation accuracy."""
    best_val_acc = 0
    train_losses, val_losses, train_accs, val_accs = [], [], [], []
    print(f"\nTraining {name}...")

    for epoch in range(epochs):
        # --- Training phase ---
        model.train()
        running_loss, correct, total = 0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / len(train_loader)
        train_acc  = correct / total
        train_losses.append(train_loss)
        train_accs.append(train_acc)

        # --- Validation phase ---
        model.eval()
        val_loss, correct, total = 0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, predicted = outputs.max(1)
                correct += predicted.eq(labels).sum().item()
                total += labels.size(0)

        val_loss = val_loss / len(val_loader)
        val_acc  = correct / total
        val_losses.append(val_loss)
        val_accs.append(val_acc)
        scheduler.step()

        # Save best checkpoint
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), save_path)

        print(f"Epoch {epoch+1}/{epochs} | Train Acc: {train_acc:.2%} | Val Acc: {val_acc:.2%}")

    print(f"Best Val Accuracy: {best_val_acc:.2%} — saved to {save_path}")
    return train_losses, val_losses, train_accs, val_accs

### 5a. EfficientNet-B0 (10 epochs)

EfficientNet-B0 is a lightweight but powerful CNN pretrained on ImageNet. We replace its classification head with a 100-class output layer and fine-tune the full network.

In [ ]:
eff_model     = timm.create_model('efficientnet_b0', pretrained=True, num_classes=100).to(device)
eff_optimizer = optim.AdamW(eff_model.parameters(), lr=1e-3, weight_decay=1e-4)
eff_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(eff_optimizer, T_max=10)

eff_train_losses, eff_val_losses, eff_train_accs, eff_val_accs = train_and_save(
    eff_model, eff_optimizer, eff_scheduler, EFFICIENTNET_PATH, epochs=10, name="EfficientNet-B0"
)

### 5b. ResNet50 (3 epochs)

ResNet50 is a deeper residual network also pretrained on ImageNet. We train it for 3 epochs to compare performance and training efficiency against EfficientNet-B0.

In [ ]:
res_model     = timm.create_model('resnet50', pretrained=True, num_classes=100).to(device)
res_optimizer = optim.AdamW(res_model.parameters(), lr=1e-3, weight_decay=1e-4)
res_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(res_optimizer, T_max=3)

res_train_losses, res_val_losses, res_train_accs, res_val_accs = train_and_save(
    res_model, res_optimizer, res_scheduler, RESNET_PATH, epochs=3, name="ResNet50"
)

## 6. Training Curves

Training and validation loss/accuracy curves allow us to monitor learning progress and detect overfitting. We observe that train accuracy slightly exceeds validation accuracy in later epochs, indicating mild overfitting — expected given the small validation set size (500 images).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(eff_train_losses, label='Train Loss')
axes[0].plot(eff_val_losses,   label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('EfficientNet-B0: Loss Curves')
axes[0].legend()

axes[1].plot(eff_train_accs, label='Train Accuracy')
axes[1].plot(eff_val_accs,   label='Val Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('EfficientNet-B0: Accuracy Curves')
axes[1].legend()

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/training_curves.png', dpi=150)
plt.show()
print("Saved to Google Drive.")

## 7. Model Evaluation

We evaluate both models on the held-out test set using three metrics: accuracy, weighted F1 score, and weighted precision/recall. Using multiple metrics is important for multi-class classification tasks where class imbalance can skew accuracy alone.

In [ ]:
def evaluate_model(model, checkpoint_path, loader):
    """Load a checkpoint and evaluate on a DataLoader, returning predictions and labels."""
    model.load_state_dict(torch.load(checkpoint_path))
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())
    return np.array(all_preds), np.array(all_labels)

# Evaluate EfficientNet-B0
eff_preds, eff_labels = evaluate_model(eff_model, EFFICIENTNET_PATH, test_loader)
eff_acc  = (eff_preds == eff_labels).mean()
eff_f1   = f1_score(eff_labels, eff_preds, average='weighted')
eff_prec = precision_score(eff_labels, eff_preds, average='weighted')
eff_rec  = recall_score(eff_labels, eff_preds, average='weighted')

# Evaluate ResNet50
res_preds, res_labels = evaluate_model(res_model, RESNET_PATH, test_loader)
res_acc  = (res_preds == res_labels).mean()
res_f1   = f1_score(res_labels, res_preds, average='weighted')
res_prec = precision_score(res_labels, res_preds, average='weighted')
res_rec  = recall_score(res_labels, res_preds, average='weighted')

# Print comparison table
print("=" * 65)
print(f"{'Model':<20} {'Epochs':>6} {'Accuracy':>10} {'Precision':>10} {'Recall':>8} {'F1':>8}")
print("=" * 65)
print(f"{'Random Baseline':<20} {'—':>6} {baseline_acc:>10.2%} {'N/A':>10} {'N/A':>8} {'N/A':>8}")
print(f"{'EfficientNet-B0':<20} {'10':>6} {eff_acc:>10.2%} {eff_prec:>10.2%} {eff_rec:>8.2%} {eff_f1:>8.2%}")
print(f"{'ResNet50':<20} {'3':>6} {res_acc:>10.2%} {res_prec:>10.2%} {res_rec:>8.2%} {res_f1:>8.2%}")
print("=" * 65)

## 8. Error Analysis

We analyze failure cases to understand what types of inputs challenge the model. Visualizing misclassified examples reveals patterns in the model's errors and suggests potential improvements.

In [ ]:
# Use the best-performing model for error analysis
best_preds  = eff_preds if eff_acc >= res_acc else res_preds
best_labels = eff_labels

misclassified = np.where(best_preds != best_labels)[0]
print(f"Total misclassified: {len(misclassified)} / {len(best_labels)} ({len(misclassified)/len(best_labels):.1%} error rate)")

# Display transform (no normalization for visualization)
display_dataset = datasets.ImageFolder(test_dir, transform=transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
]))

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
axes = axes.flatten()

for i, idx in enumerate(misclassified[:12]):
    image, _ = display_dataset[idx]
    image_np = image.permute(1, 2, 0).numpy()
    true_label = class_names[best_labels[idx]]
    pred_label = class_names[best_preds[idx]]
    axes[i].imshow(image_np)
    axes[i].set_title(f"True: {true_label}\nPred: {pred_label}", fontsize=8, color='red')
    axes[i].axis('off')

plt.suptitle('Misclassified Examples', fontsize=14)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/error_analysis.png', dpi=150)
plt.show()

In [ ]:
# Most confused class pairs
confused_pairs = [
    (class_names[t], class_names[p])
    for t, p in zip(best_labels[misclassified], best_preds[misclassified])
]
most_confused = Counter(confused_pairs).most_common(10)

print("Most confused class pairs (true → predicted):")
print("-" * 55)
for (true, pred), count in most_confused:
    print(f"  {true:25s} → {pred:25s} ({count}x)")

print("\nAnalysis: Most confusions occur between visually similar sports")
print("(e.g. sidecar ↔ motorcycle racing, snow boarding ↔ giant slalom.")
print("These are reasonable errors that even human viewers might make.")

## 9. Web Application Deployment

We deploy the best model as an interactive web application using Gradio. Users can upload any sports image and receive the top 5 predicted sport categories with confidence scores.

In [1]:
import gradio as gr

# Load the best model
best_model_path = EFFICIENTNET_PATH if eff_acc >= res_acc else RESNET_PATH
best_model      = eff_model if eff_acc >= res_acc else res_model
best_model.load_state_dict(torch.load(best_model_path))
best_model.eval()

def predict(image):
    """Run inference on a PIL image and return top-5 class probabilities."""
    img = val_test_transforms(image).unsqueeze(0).to(device)
    with torch.no_grad():
        outputs = best_model(img)
        probs   = torch.softmax(outputs, dim=1)
        top5    = probs.topk(5)
    return {class_names[idx]: float(prob)
            for prob, idx in zip(top5.values[0], top5.indices[0])}

demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil"),
    outputs=gr.Label(num_top_classes=5),
    title="Sports Image Classifier",
    description="Upload a sports image and the model will predict which of 100 sports it is!",
)

demo.launch(share=True)

NameError: name 'eff_acc' is not defined